# Entrega Final - TURRI Valentín | Data Mining


En primer lugar, se importan las librerías necesarias.

In [ ]:
import pandas as pd

import sklearn as sk
from sklearn import model_selection
from sklearn import ensemble
from sklearn import metrics

Se monta Google Drive dentro del entorno Colab.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


## 0. Lectura de datos

Se lee el Data Frame con el cuál se trabajara y el archivo que contiene los datos cuyos precios tienen que determinarse.

In [ ]:
DIR = "/content/drive/MyDrive/UBA/Data Mining"

In [ ]:
df_ent = pd.read_csv(f"{DIR}/entrenamiento.csv", index_col="id")

In [ ]:
df_ap = pd.read_csv(f"{DIR}/a_predecir.csv", index_col="id")

## 1. Entender los datos (AID)

## 2. Limpiar y transformar los datos (MD)

Se seleccionan las filas dónde el precio no sea nulo.
Se filtran los datos por aquellos que corresponden solo a Capital Federal.
Se consideran operaciones solo del tipo Venta.
Además, se consideran solo las propiedades del tipo Cochera, Casa o Departamento.

In [ ]:
# selección de datos
df_ent = df_ent.loc[(df_ent["price"].notna())  &  ## & (df_ent["currency"] == "USD")
    (df_ent["l2"] == "Capital Federal") & (df_ent["operation_type"] == 'Venta') & \
    (df_ent["property_type"].isin(["Cochera", "Departamento", "Casa"]))]
df_ent.shape

(105489, 24)

Los datos cuyos precios están en Pesos Argentinos son convertidos a Dolares con el tipo de cambio de 2020.

In [ ]:
PESOS_POR_DOLAR = 78.5
df_ent.loc[df_ent["currency"] == "ARS", "price"] = df_ent.loc[df_ent["currency"] == "ARS", "price"] / PESOS_POR_DOLAR

Se concatenan df_ent y df_ap

In [ ]:
df = pd.concat([df_ent, df_ap])

Para las propiedades del tipo Cochera se considera que estas no tienen baños, cuartos y dormitorios.

In [ ]:
# las cocheras no tienen bathrooms, rooms o bedrooms
df.loc[df["property_type"] == "Cochera", ["bathrooms", "rooms", "bedrooms"]] = 0

Imputación de valores faltantes en columnas relacionadas: se completan los dormitorios en base a los ambientes, y viceversa; se estima la cantidad de baños como la diferencia entre ambientes y dormitorios. También se completan las superficies total y cubierta mutuamente si falta una de ellas, y se calcula el porcentaje de superficie cubierta sobre el total.
```


In [ ]:
df.loc[:, "bedrooms"] = df["bedrooms"].fillna(df["rooms"] - 1)
df.loc[:, "rooms"] = df["rooms"].fillna(df["bedrooms"] + 1)
df.loc[:, "bathrooms"] = df["bathrooms"].fillna(df["rooms"] - df["bedrooms"])

df["surface_total"] = df["surface_total"].fillna(df["surface_covered"])
df["surface_covered"] = df["surface_covered"].fillna(df["surface_total"])
df["pct_surface_covered"] = df["surface_covered"] / df["surface_total"]

Se conservan solo las propiedades que no tienen precio o que sí tienen superficie total informada.

In [ ]:
# dejo las que no tienen precio o tienen superficie total
df = df.loc[df["price"].isna() | df["surface_total"].notna()]
df.shape

(87583, 25)

Se imputa ''Distrito Audiovisual'' por ''Chacarita'' y ''Catalinas'' por ''Retiro''.

In [ ]:
# "limpieza" de l3 e imputación de l4
df.loc[df["l3"] == "Distrito Audiovisual", "l3"] = "Chacarita"
df.loc[df["l3"] == "Catalinas", "l3"] = "Retiro"
df.loc[:, "l4"] = df["l4"].fillna(df["l3"])

Se calcula el precio por metro cuadrado, evitando división por cero al sumar una pequeña constante al total de superficie.

In [ ]:
df_ent.loc[:, "price_m2"] = df_ent["price"] / (df_ent["surface_total"] + 0.001)

Se pone un rango para evitar considerar outliers con respecto a precio por m2

In [ ]:
# filtro lo que tiene un precio por metro cuadrado sin sentido para calcular la estadística
df_ent = df_ent[(df_ent["price_m2"] >= 500) & (df_ent["price_m2"] <= 10000)]
df_ent.shape

(75439, 25)

Se completan los valores faltantes en la ubicación l4 usando l3, y luego se calcula el precio promedio por metro cuadrado agrupado por tipo de propiedad y ubicación.

In [ ]:
df_ent.loc[:, "l4"] = df_ent["l4"].fillna(df_ent["l3"])
gb = df_ent.groupby(by=["property_type", "l4"])["price_m2"].mean().reset_index()
gb

,property_type,l4,price_m2
0,Casa,Abasto,1638.860004
1,Casa,Agronomía,1639.703726
2,Casa,Almagro,1167.648521
3,Casa,Balvanera,1563.899913
4,Casa,Barracas,1249.043058
...,...,...,...
167,Departamento,Villa Riachuelo,1551.667721
168,Departamento,Villa Santa Rita,2182.146642
169,Departamento,Villa Soldati,863.765979
170,Departamento,Villa Urquiza,2830.510975


Se incorporan los precios promedio por m² según tipo de propiedad y ubicación mediante un merge, y luego se estima el precio total multiplicando ese valor por la superficie total.

In [ ]:
df = df.reset_index().merge(gb, on=["property_type", "l4"], how="left").set_index("id")
df["price_m2_x_surface_total"] = df["price_m2"] * df["surface_total"]

In [ ]:
df

,ad_type,start_date,end_date,created_on,lat,lon,l1,l2,l3,l4,...,currency,price_period,title,description,property_type,operation_type,price,pct_surface_covered,price_m2,price_m2_x_surface_total
id,,,,,,,,,,,,,,,,,,,,,
521738,Propiedad,2019-08-05,2019-08-31,2019-08-05,-58.429983,-34.607225,Argentina,Capital Federal,Almagro,Almagro,...,USD,NaN,Venta 3 Ambientes - Almagro - Balcón - Ameniti...,Corredor Responsable: Marcelo Trujillo - CUCIC...,Departamento,Venta,173000.0,0.969697,2511.085728,165731.658080
499733,Propiedad,2019-07-24,9999-12-31,2019-07-24,-58.404054,-34.599623,Argentina,Capital Federal,Barrio Norte,Barrio Norte,...,USD,Mensual,Tres ambientes de categoria en Barrio Norte,Excelente y amplio departamento de tres ambien...,Departamento,Venta,190000.0,0.945055,3115.388807,283500.381465
473866,Propiedad,2019-07-06,2019-07-09,2019-07-06,-58.421650,-34.588602,Argentina,Capital Federal,Palermo,Palermo,...,USD,NaN,"DEPARTAMENTO 3 AMBIENTES EN VENTA , PALERMO",Excelente ubicación!Scalabrini Ortiz entre Par...,Departamento,Venta,240000.0,1.000000,3449.851252,275988.100157
155953,Propiedad,2019-12-03,2019-12-17,2019-12-03,-58.378095,-34.608478,Argentina,Capital Federal,Monserrat,Monserrat,...,USD,NaN,Departamento de 2 ambientes en Venta en Monserrat,"Excelente 2 Abientes frente balcon ,muy lumi...",Departamento,Venta,99000.0,0.895833,2070.692497,99393.239854
836819,Propiedad,2020-04-28,2020-05-08,2020-04-28,-58.420295,-34.606271,Argentina,Capital Federal,Almagro,Almagro,...,USD,NaN,Departamento de 3 ambientes en Venta en Almagro,"Lindo 3 ambientes con depencia, al frente con ...",Departamento,Venta,190000.0,0.950000,2511.085728,200886.858279
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
960218,Propiedad,2020-05-30,2020-06-01,2020-05-30,-58.420206,-34.602580,Argentina,Capital Federal,Almagro,Almagro,...,USD,NaN,Departamento de 4 ambientes en Venta en Almagro,4 AMBIENTES C-DEPENDENCIA. MUY BUEN ESTADO.MUE...,Departamento,Venta,NaN,0.884615,2511.085728,261152.915763
960220,Propiedad,2020-05-30,2020-06-01,2020-05-30,-58.411129,-34.578322,Argentina,Capital Federal,Palermo,Palermo,...,USD,NaN,VENTA semipiso 3 dormit y dependencia en excel...,"LIVING / COMEDOR, TOILETTE, 3 DORMITORIOS, 1 B...",Departamento,Venta,NaN,1.000000,3449.851252,344985.125196
960226,Propiedad,2020-05-30,2020-06-01,2020-05-30,-58.462594,-34.572645,Argentina,Capital Federal,Belgrano,Belgrano,...,USD,NaN,VENTA PISO ALTO con entrada privada al CLUB BE...,"GASTOS: Expensas ; ABL: LIVING / COMEDOR, BAL...",Departamento,Venta,NaN,1.000000,3442.655217,826237.252173


Se hace filtrado

In [ ]:
# filtro las que difieren en más de 1000 (por encima) del precio de la estadística
p = (df["price"] / (df["surface_total"] + 0.001) - df["price_m2"]) > 1000
df = df[~p]

In [ ]:
df = pd.get_dummies(df, columns=["property_type", "l4"], dtype="uint8")

In [ ]:
df[df.columns.drop("price")] = df[df.columns.drop("price")].fillna(0.0)

<ipython-input-28-3909636559>:1: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df[df.columns.drop("price")] = df[df.columns.drop("price")].fillna(0.0)


Se llenan los valores vacíos de Latitud y Longitud con la Moda

In [ ]:
# Función auxiliar que rellena NA con la moda del grupo (el primer valor en caso de haber varias)
def fillna_with_mode(s):
    mode_val = s.mode()
    return s.fillna(mode_val.iloc[0] if not mode_val.empty else s)

# Rellenamos los NA en 'lat' agrupando por barrio (columna "l3")
df_ent['lat'] = df_ent.groupby('l3')['lat'].transform(fillna_with_mode)

# Rellenamos los NA en 'lon' agrupando por barrio (columna "l3")
df_ent['lon'] = df_ent.groupby('l3')['lon'].transform(fillna_with_mode)

# Verificamos los cambios
print(df_ent[['l3', 'lat', 'lon']].head())

                  l3        lat        lon
id                                        
521738       Almagro -58.429983 -34.607225
499733  Barrio Norte -58.404054 -34.599623
473866       Palermo -58.421650 -34.588602
155953     Monserrat -58.378095 -34.608478
836819       Almagro -58.420295 -34.606271


Se hace lo mismo para el DF a predecir

In [ ]:
# Función auxiliar que rellena NA con la moda del grupo (el primer valor en caso de haber varias)
def fillna_with_mode(s):
    mode_val = s.mode()
    return s.fillna(mode_val.iloc[0] if not mode_val.empty else s)

# Rellenamos los NA en 'lat' agrupando por barrio (columna "l3")
df_ap['lat'] = df_ap.groupby('l3')['lat'].transform(fillna_with_mode)

# Rellenamos los NA en 'lon' agrupando por barrio (columna "l3")
df_ap['lon'] = df_ap.groupby('l3')['lon'].transform(fillna_with_mode)

# Verificamos los cambios
print(df_ap[['l3', 'lat', 'lon']].head())

                    l3        lat        lon
id                                          
1068           Palermo -58.419773 -34.597480
1069           Palermo -58.426576 -34.590987
1073          Floresta -58.479808 -34.631266
1082      Villa Crespo -58.437889 -34.603291
1091  Villa del Parque -58.476461 -34.602494


## 3. Entrenamiento del modelos (AA)

In [ ]:
# vuelvo a partir los datos
df = df.select_dtypes(['number', 'bool'])
df_ent = df[df["price"].notna()]
df_ap = df[df["price"].isna()]
df_ent.shape, df_ap.shape

((73344, 76), (7012, 76))

In [ ]:
X = df_ent[df_ent.columns.drop('price')]
y = df_ent['price']

Se utiliza el Gradient Boosting Rregressor

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingRegressor
from sklearn import metrics
import numpy as np

# División de datos
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

best_score_test = float('inf')
best_params = {}

# Grid search con GradientBoostingRegressor
for n_estimators in [10, 50, 100, 300, 500, 700, 1000]:
    for max_depth in [3, 5, 8, 10, 12]:
        # Modelo habilitado
        reg = GradientBoostingRegressor(
            n_estimators=n_estimators,
            max_depth=max_depth,
            min_samples_leaf=10,
            min_samples_split=20,
            random_state=42
        )
        # Modelo deshabilitado
        # reg = RandomForestRegressor(...)

        # Entreno
        reg.fit(X_train, y_train)

        # Predicciones
        y_pred_train = reg.predict(X_train)
        y_pred_test  = reg.predict(X_test)

        # RMSE manual
        rmse_train = np.sqrt(metrics.mean_squared_error(y_train, y_pred_train))
        rmse_test  = np.sqrt(metrics.mean_squared_error(y_test,  y_pred_test))

        # Actualizo mejor score
        if rmse_test < best_score_test:
            best_score_test = rmse_test
            best_params = {
                'n_estimators': n_estimators,
                'max_depth': max_depth
            }

        print(
            f"n_estimators={n_estimators} | max_depth={max_depth} -> "
            f"RMSE train={rmse_train:.2f}, test={rmse_test:.2f} "
            f"(best={best_score_test:.2f})"
        )

print("Mejores parámetros encontrados:", best_params)
print(f"Mejor RMSE en test: {best_score_test:.2f}")

n_estimators=10 | max_depth=3 -> RMSE train=135162.43, test=136956.28 (best=136956.28)
n_estimators=10 | max_depth=5 -> RMSE train=119445.09, test=115826.97 (best=115826.97)
n_estimators=10 | max_depth=8 -> RMSE train=109718.00, test=107928.43 (best=107928.43)
n_estimators=10 | max_depth=10 -> RMSE train=106809.58, test=106184.27 (best=106184.27)
n_estimators=10 | max_depth=12 -> RMSE train=103687.77, test=104560.11 (best=104560.11)
n_estimators=50 | max_depth=3 -> RMSE train=88442.03, test=79484.21 (best=79484.21)
n_estimators=50 | max_depth=5 -> RMSE train=77016.87, test=73906.47 (best=73906.47)
n_estimators=50 | max_depth=8 -> RMSE train=62756.90, test=70727.82 (best=70727.82)
n_estimators=50 | max_depth=10 -> RMSE train=56073.31, test=68082.30 (best=68082.30)
n_estimators=50 | max_depth=12 -> RMSE train=51316.91, test=67246.74 (best=67246.74)
n_estimators=100 | max_depth=3 -> RMSE train=81331.46, test=76069.88 (best=67246.74)
n_estimators=100 | max_depth=5 -> RMSE train=71898.96, t

## 4. Solución para subir Kaggle

In [ ]:
df_ap.shape

(7012, 76)

In [ ]:
X = df_ent[df_ent.columns.drop('price')]
y = df_ent['price']

n_estimators = 50
max_depth = 15

reg = GradientBoostingRegressor(
    n_estimators=n_estimators,
    max_depth=max_depth,
    min_samples_leaf=10,
    min_samples_split=20,
    learning_rate=0.1,  # parámetro adicional en boosting
    random_state=42
)

# Entrenamos el modelo con todos los datos
reg.fit(X, y)

GradientBoostingRegressor(max_depth=15, min_samples_leaf=10,
                          min_samples_split=20, n_estimators=50,
                          random_state=42)

In [ ]:
X_ap = df_ap[X.columns]

# Predecimos los precios del dataset a predecir
y_pred_ap = reg.predict(X_ap)
y_pred_ap.shape

(7012,)

In [ ]:
# Lleno el precio de df_ap con las predicciones
df_ap["price"] = y_pred_ap

# Grabo el df_ap en un archivo csv para subir a Kaggle
df_ap["price"].to_csv("Entrega Final - TURRI Valentín.csv")

<ipython-input-37-2422717627>:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_ap["price"] = y_pred_ap


In [ ]:
# 2) Importa el helper de Colab y lanza la descarga
from google.colab import files
files.download("Entrega Final - TURRI Valentín.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>